# Overview Prático: Limpeza e Tratamento de Dados com Pandas

A **limpeza de dados (*Data Cleaning*)** é a etapa em que identificamos e corrigimos erros, incoerências, valores nulos e discrepâncias em uma base bruta. Nenhum modelo estatístico ou de *Machine Learning* gera bons resultados se for alimentado com dados de baixa qualidade (*Garbage in, Garbage out*).

---

### Objetivos deste Guia:
1. **Inspeção estrutural:** mapear dimensões, tipos e integridade da base.
2. **Tratamento de duplicatas:** remover registros redundantes.
3. **Gestão de nulos (*missing values*):** entender estratégias de remoção e imputação.
4. **Adequação de tipos (*casting*):** converter representações de texto em tipos numéricos e temporais.
5. **Padronização textual:** tratar inconsistências ortográficas, caixa e espaços extras.
6. **Análise de anomalias (*outliers*):** identificar e tratar valores extremos visual e estatisticamente.

A seguir, criamos uma massa de dados de teste contendo intencionalmente os problemas mais recorrentes do dia a dia de um analista:

In [ ]:
import pandas as pd
import numpy as np

data = {
    'produto': ['  Produto A ', 'Produto B', 'Produto C', 'Produto A', 'Produto D', 'Produto E', 'Produto E', None],
    'data_venda': ['2025-10-01', '2025-10-02', '2025-10-03', '2025-10-01', '2025-10-05', '2025-10-06','2025-10-06', '2025-10-07'],
    'valor': ['R$ 50,50', 'R$ 70,00', 'R$ 30,25', 'R$ 50,50', 'R$ 1500,00', None, None, 'R$ 90,10'],
    'estoque': [100, 50, None, 100, 5, 200, 200, 10],
    'cidade': ['São Paulo', 'sp', 'Rio de Janeiro', 'São Paulo', 'Belo Horizonte', 'são paulo', 'são paulo', 'Recife']
}
df = pd.DataFrame(data)
df

---
## Passo 0: Inspeção e Diagnóstico Inicial

Antes de aplicar qualquer transformação, é indispensável diagnosticar o estado dos dados. A inspeção inicial serve para responder:
* Quantos registros e colunas existem?
* Quais são os tipos de dados inferidos inicialmente?
* Onde estão as anomalias visíveis e campos nulos?

> **Métodos principais:**
> - `df.head()`: amostra inicial dos registros.
> - `df.info()`: contagem de não-nulos e tipos primitivos de memória.
> - `df.describe()`: estatísticas descritivas das variáveis numéricas.

In [ ]:
df.head()

In [ ]:
# Obter informações gerais da base (tipos de dados e contagem de não-nulos)
df.info()

In [ ]:
# Resumo estatístico das variáveis numéricas
df.describe()

---
## Passo 1: Detecção e Remoção de Registros Duplicados

Linhas inteiramente duplicadas geram contagens infladas, distorcem métricas de agregação e causam vazamento de dados em etapas de modelagem preditiva.

### Fluxo de trabalho:
1. Identificar o volume de linhas idênticas com `.duplicated().sum()`.
2. Decidir se o descarte deve considerar a linha inteira ou apenas um subconjunto de chaves (`subset=[...]`).
3. Aplicar o descarte preservando o primeiro ou último registro com `.drop_duplicates(keep='first')`.

In [ ]:
# Verificando a quantidade de linhas duplicadas
df.duplicated().sum()

In [ ]:
# Removendo as linhas duplicadas mantendo a primeira ocorrência
df = df.drop_duplicates(keep='first')

In [ ]:
# Confirmando se ainda existem duplicatas
print(f"Duplicatas restantes: {df.duplicated().sum()}")

In [ ]:
# Visualizando o DataFrame atualizado após remoção de duplicatas
df

---
## Passo 2: Gestão de Dados Ausentes (*Missing Values*)

Valores nulos (`NaN` / `None`) surgem por falhas de integração, preenchimento opcional ou perda na captura.

Existem duas abordagens clássicas:
* **Exclusão (`.dropna()`):** recomendada quando a proporção de nulos é insignificante ou quando a linha inteira perde a utilidade de negócio sem aquele dado.
* **Imputação (`.fillna()`):** preenchimento por valores fixos, categóricos padrão (`"desconhecido"`) ou estatísticos (média, mediana ou moda).

Abaixo, verificamos primeiro a distribuição dos nulos por coluna:

In [ ]:
# Contagem de valores nulos por coluna
df.isnull().sum()

### Exemplo de Exclusão Completa
Se quiséssemos descartar qualquer linha com pelo menos um campo nulo, utilizaríamos `df.dropna()`. Vamos testar isso separadamente sem alterar o nosso DataFrame principal:

In [ ]:
# Exemplo: remove todas as linhas com ao menos um valor nulo
df_sem_nulos = df.dropna()
df_sem_nulos

### Aplicando Estratégias Mistas de Limpeza no DataFrame:
* **Estoque (Numérico contínuo):** imputação pela **mediana**, tornando a métrica resistente a eventuais distorções por valores extremos.
* **Produto (Categórico):** imputação com a classe genérica `'desconhecido'`.
* **Valor (Chave crítica de negócio):** descarte da linha específica via `.dropna(subset=['valor'])`, já que uma venda sem valor compromete métricas financeiras.

In [ ]:
# 1. Imputação por mediana em 'estoque'
mediana_estoque = df['estoque'].median()
df['estoque'] = df['estoque'].fillna(mediana_estoque)

# 2. Imputação categórica em 'produto'
df['produto'] = df['produto'].fillna('desconhecido')

# 3. Descarte de linhas com 'valor' ausente
df = df.dropna(subset=['valor'])

In [ ]:
# Visualizando o DataFrame após o tratamento dos dados ausentes
df

---
## Passo 3: Conversão e Adequação de Tipos de Dados (*Type Casting*)

Muitas vezes colunas numéricas ou datas chegam tipadas genericamente como texto (`object` / `string`) por conterem símbolos monetários, espaços ou caracteres regionais (como vírgula para casas decimais).

### Ações necessárias:
* **`valor`:** remover o prefixo `'R$'`, substituir a vírgula regional por ponto (`.`) e converter para formato numérico decimal (`float64`) via `pd.to_numeric()`.
* **`data_venda`:** converter a cadeia de texto no tipo nativo temporal via `pd.to_datetime()`, habilitando operações de série temporal.

In [ ]:
# Tratamento da coluna de valores monetários
df['valor'] = df['valor'].str.replace('R$', '', regex=False).str.replace(',', '.').str.strip()
df['valor'] = pd.to_numeric(df['valor'])

In [ ]:
# Conversão da coluna de data de string para datetime
df['data_venda'] = pd.to_datetime(df['data_venda'])

# Verificando a nova estrutura de tipos de dados
df.info()

In [ ]:
# Visualizando o DataFrame com os tipos corrigidos
df

---
## Passo 4: Padronização e Higienização de Strings

Inconsistências em cadeias de texto fragmentam categorias idênticas em grupos distintos em análises de agrupamento (`groupby`) ou filtros.

### Problemas comuns e soluções:
1. **Espaços residuais (`.str.strip()`):** remove espaços em branco acidentais antes e depois do texto.
2. **Caixa alta/baixa desordenada (`.str.lower()` ou `.str.upper()`):** equaliza variações como `"São Paulo"`, `"são paulo"` e `"SÃO PAULO"`.
3. **Erros de digitação e abreviações (`.replace()` / `.loc`):** unifica variações como `"sp"` para `"são paulo"`.

In [ ]:
# 1. Removendo espaços em branco residuais no início e no final
df['produto'] = df['produto'].str.strip()
df

In [ ]:
# 2. Padronizando cidades para caixa baixa (minúsculas)
df['cidade'] = df['cidade'].str.lower()
df

In [ ]:
# 3. Mapeando e unificando abreviações ou sinônimos
mapeamento = {'sp': 'são paulo'}
condicao = df['cidade'] == 'sp'
df.loc[condicao, 'cidade'] = df.loc[condicao, 'cidade'].replace(mapeamento)

df

---
## Passo 5: Detecção e Análise Visual de Outliers

*Outliers* são observações que se distanciam drasticamente do padrão da distribuição. Podem decorrer de erros operacionais (ex: digitação de zeros a mais) ou representar fenômenos reais e raros.

A primeira abordagem é sempre a **inspeção gráfica**:
* O **Boxplot** é o gráfico ideal para visualizar a dispersão, a mediana e identificar pontos que ultrapassam os limites convencionais de distribuição.

In [ ]:
import matplotlib.pyplot as plt

# Visualização com Boxplot para as colunas numéricas
df[['valor', 'estoque']].plot(kind='box', subplots=True, layout=(1, 2), figsize=(8, 4))
plt.tight_layout()
plt.show()

---
## Passo 6: Detecção e Tratamento Estatístico via Intervalo Interquartil (IQR)

Para regras dinâmicas e independentes de premissas de normalidade estrita da curva, adota-se o **Método do Intervalo Interquartil (IQR)**:

$$\text{IQR} = Q_3 - Q_1$$

### Limites de corte:
* **Limite Inferior:** $Q_1 - 1.5 \times \text{IQR}$
* **Limite Superior:** $Q_3 + 1.5 \times \text{IQR}$

Qualquer registro além desses limites é formalmente sinalizado como candidato a descarte, investigação detalhada ou transformação matemática (como logaritmo).

In [ ]:
# Calculando os quartis e o IQR da coluna 'valor'
Q1 = df['valor'].quantile(0.25)
Q3 = df['valor'].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print(f"Q1: {Q1:.2f} | Q3: {Q3:.2f} | IQR: {IQR:.2f}")
print(f"Limite Inferior: {limite_inferior:.2f}")
print(f"Limite Superior: {limite_superior:.2f}")

# Identificando os registros considerados outliers
outliers = df[(df['valor'] < limite_inferior) | (df['valor'] > limite_superior)]
print("\nOutliers encontrados:")
outliers

### Ação sobre o Outlier:
Após confirmar estatisticamente que a venda de `R$ 1500,00` excede o limite superior do padrão das transações deste lote, filtramos o conjunto retendo apenas os valores considerados dentro do limite operacional aceito.

In [ ]:
# Filtrando e mantendo apenas registros dentro do limite considerado aceitável
df = df[df['valor'] <= 1000]
df

---
## Resumo e Checklist do Fluxo de Limpeza

Sempre que iniciar um projeto de dados, siga este roteiro de verificação:

| Etapa | Ferramentas Típicas | Objetivo Principal |
| :--- | :--- | :--- |
| **0. Inspeção** | `.head()`, `.info()`, `.describe()` | Reconhecimento de estrutura, tipos e dimensões. |
| **1. Duplicatas** | `.duplicated()`, `.drop_duplicates()` | Garantia de unicidade dos registros. |
| **2. Nulos** | `.isna()`, `.fillna()`, `.dropna()` | Resolução de lacunas por descarte ou imputação estatística. |
| **3. Tipagem** | `pd.to_numeric()`, `pd.to_datetime()`, `.astype()` | Conversão correta para cálculos e filtros. |
| **4. Strings** | `.str.strip()`, `.str.lower()`, `.replace()` | Eliminação de ruídos e unificação categórica. |
| **5. Outliers** | `Boxplot`, Método IQR | Identificação e tratamento de anomalias extremas. |

Com estes passos executados com rigor metodológico, sua base de dados atinge a consistência necessária para etapas posteriores de Análise Exploratória (EDA) e Modelagem Preditiva.